# NSE RAG Agent (Notebook version)

Local, on-prem RAG system over NSE corporate filings + news, using **Ollama (Llama 3.1)**
for generation and **Chroma** for vector storage. Nothing leaves your machine.

**Before running:** install Ollama (https://ollama.com), then in a terminal:
```
ollama pull llama3.1:8b
ollama pull nomic-embed-text
```

Run the cells top to bottom, in order:
1. Setup / imports
2. Ingest filings (NSE corporate announcements)
3. Ingest news (Google News RSS)
4. Inspect what was pulled
5. Build the vector store (chunk + embed)
6. Query the RAG agent


## 0. Setup
Install dependencies (run once) and create the project folders.

In [1]:
# Run once -- installs everything needed
%pip install chromadb ollama requests feedparser beautifulsoup4 python-dateutil tqdm -q


Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install --upgrade pip

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import json
import time
from pathlib import Path

import requests
import feedparser
from dateutil import parser as dateparser
import chromadb
import ollama
from tqdm import tqdm

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
FILINGS_DIR = DATA_DIR / "filings"
NEWS_DIR = DATA_DIR / "news"
CHROMA_DIR = PROJECT_DIR / "chroma_db"

for d in (FILINGS_DIR, NEWS_DIR, CHROMA_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)

Project directory: c:\Users\ishit\Downloads


## 1. Ingest NSE filings/announcements

NSE's website blocks plain requests without browser-like headers and a warmed-up
session cookie -- `get_session()` mimics that handshake. NSE also rate-limits
aggressively, so there's a sleep between calls.

**If you get repeated 401/403 errors:** open the corporate-announcements page on
nseindia.com in a browser, check dev tools (Network tab) for the current API path
and params, and update `ANNOUNCEMENTS_ENDPOINT` below.

In [2]:
BASE_URL = "https://www.nseindia.com"
ANNOUNCEMENTS_ENDPOINT = "https://www.nseindia.com/api/corporate-announcements"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nseindia.com/companies-listing/corporate-filings-announcements",
}


def get_session() -> requests.Session:
    """NSE requires hitting the homepage first to pick up cookies
    before the API responds with real data instead of a 401/403."""
    s = requests.Session()
    s.headers.update(HEADERS)
    s.get(BASE_URL, timeout=10)
    time.sleep(1)
    return s


def fetch_announcements(session: requests.Session, symbol: str) -> list[dict]:
    params = {"index": "equities", "symbol": symbol}
    resp = session.get(ANNOUNCEMENTS_ENDPOINT, params=params, timeout=10)
    if resp.status_code != 200:
        print(f"[warn] {symbol}: HTTP {resp.status_code} -- NSE may be rate-limiting you")
        return []
    try:
        data = resp.json()
    except ValueError:
        print(f"[warn] {symbol}: non-JSON response, skipping")
        return []

    records = []
    for item in data:
        records.append({
            "ticker": symbol,
            "title": item.get("desc", "").strip(),
            "description": item.get("attchmntText", "").strip(),
            "date": item.get("an_dt", ""),
            "source_url": item.get("attchmntFile", ""),
        })
    return records

from datetime import datetime, timedelta

def filter_last_year(records: list[dict]) -> list[dict]:
    """Keep only records from the last 365 days."""
    cutoff = datetime.now() - timedelta(days=365)
    kept = []
    for r in records:
        raw_date = r.get("date", "")
        if not raw_date:
            continue
        try:
            parsed = datetime.strptime(raw_date.split()[0], "%d-%b-%Y")
        except ValueError:
            continue
        if parsed >= cutoff:
            kept.append(r)
    return kept

In [3]:
TICKERS = ["RELIANCE", "TCS", "INFY", "HDFCBANK"]

session = get_session()
for symbol in TICKERS:
    print(f"Fetching filings for {symbol}...")
    records = fetch_announcements(session, symbol)
    records = filter_last_year(records)
    out_path = FILINGS_DIR / f"{symbol}.json"
    with open(out_path, "w") as f:
        json.dump(records, f, indent=2)
    print(f"  -> saved {len(records)} records to {out_path}")
    time.sleep(2)

Fetching filings for RELIANCE...
  -> saved 169 records to c:\Users\ishit\Downloads\data\filings\RELIANCE.json
Fetching filings for TCS...
  -> saved 194 records to c:\Users\ishit\Downloads\data\filings\TCS.json
Fetching filings for INFY...
  -> saved 231 records to c:\Users\ishit\Downloads\data\filings\INFY.json
Fetching filings for HDFCBANK...
  -> saved 149 records to c:\Users\ishit\Downloads\data\filings\HDFCBANK.json


In [4]:
test_records = fetch_announcements(session, "RELIANCE")
print(len(test_records))
print(test_records[0]["date"] if test_records else "no records")

3317
10-Jul-2026 17:46:25


## 2. Ingest news

Uses Google News RSS (free, no API key) filtered per company.

In [5]:
GOOGLE_NEWS_RSS = "https://news.google.com/rss/search?q={query}+NSE+stock&hl=en-IN&gl=IN&ceid=IN:en"


def fetch_news(company_name: str, ticker: str, max_items: int = 25) -> list[dict]:
    url = GOOGLE_NEWS_RSS.format(query=company_name.replace(" ", "+"))
    feed = feedparser.parse(url)

    records = []
    for entry in feed.entries[:max_items]:
        try:
            date = dateparser.parse(entry.get("published", "")).isoformat()
        except (ValueError, TypeError):
            date = ""
        records.append({
            "ticker": ticker,
            "title": entry.get("title", "").strip(),
            "summary": entry.get("summary", "").strip(),
            "date": date,
            "source_url": entry.get("link", ""),
        })
    return records

In [6]:
# Edit to match your tracked tickers -- full company name improves search recall
COMPANIES = {
    "RELIANCE": "Reliance Industries",
    "TCS": "Tata Consultancy Services",
    "INFY": "Infosys",
    "HDFCBANK": "HDFC Bank",
}

for ticker, name in COMPANIES.items():
    print(f"Fetching news for {ticker} ({name})...")
    records = fetch_news(name, ticker)
    out_path = NEWS_DIR / f"{ticker}.json"
    with open(out_path, "w") as f:
        json.dump(records, f, indent=2)
    print(f"  -> saved {len(records)} records to {out_path}")

Fetching news for RELIANCE (Reliance Industries)...
  -> saved 25 records to c:\Users\ishit\Downloads\data\news\RELIANCE.json
Fetching news for TCS (Tata Consultancy Services)...
  -> saved 25 records to c:\Users\ishit\Downloads\data\news\TCS.json
Fetching news for INFY (Infosys)...
  -> saved 25 records to c:\Users\ishit\Downloads\data\news\INFY.json
Fetching news for HDFCBANK (HDFC Bank)...
  -> saved 25 records to c:\Users\ishit\Downloads\data\news\HDFCBANK.json


## 3. Inspect what was pulled

Worth eyeballing before spending time embedding everything -- especially since filings ingestion is the flaky part.

In [7]:
for path in sorted(FILINGS_DIR.glob("*.json")):
    with open(path) as f:
        items = json.load(f)
    print(f"{path.stem}: {len(items)} filings")

print()
for path in sorted(NEWS_DIR.glob("*.json")):
    with open(path) as f:
        items = json.load(f)
    print(f"{path.stem}: {len(items)} news items")

HDFCBANK: 149 filings
INFY: 231 filings
RELIANCE: 169 filings
TCS: 194 filings

HDFCBANK: 25 news items
INFY: 25 news items
RELIANCE: 25 news items
TCS: 25 news items


In [8]:
# Peek at a sample record
sample_path = NEWS_DIR / "RELIANCE.json"
if sample_path.exists():
    with open(sample_path) as f:
        sample = json.load(f)
    sample[:2]

## 4. Build the vector store

Chunks every filing/news record, embeds it with `nomic-embed-text` via Ollama,
and stores it in a local Chroma collection persisted to `chroma_db/`.

In [9]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
EMBED_MODEL = "nomic-embed-text"
COLLECTION_NAME = "nse_filings_news"


def chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    if not text or len(text.strip()) == 0:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start += size - overlap
    return chunks


def load_records() -> list[dict]:
    """Normalize filings + news JSON into a common shape:
    {ticker, kind, title, body, date, source_url}"""
    records = []

    for path in FILINGS_DIR.glob("*.json"):
        with open(path) as f:
            items = json.load(f)
        for item in items:
            body = f"{item.get('title', '')}\n{item.get('description', '')}".strip()
            records.append({
                "ticker": item.get("ticker", path.stem),
                "kind": "filing",
                "title": item.get("title", ""),
                "body": body,
                "date": item.get("date", ""),
                "source_url": item.get("source_url", ""),
            })

    for path in NEWS_DIR.glob("*.json"):
        with open(path) as f:
            items = json.load(f)
        for item in items:
            body = f"{item.get('title', '')}\n{item.get('summary', '')}".strip()
            records.append({
                "ticker": item.get("ticker", path.stem),
                "kind": "news",
                "title": item.get("title", ""),
                "body": body,
                "date": item.get("date", ""),
                "source_url": item.get("source_url", ""),
            })

    return records

In [10]:
records = load_records()
print(f"Loaded {len(records)} raw records (filings + news)")

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
# Fresh build each run -- delete old collection if present
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = client.create_collection(COLLECTION_NAME)

doc_id = 0
for record in tqdm(records, desc="Embedding chunks"):
    chunks = chunk_text(record["body"])
    for i, chunk in enumerate(chunks):
        if not chunk.strip():
            continue
        embedding = ollama.embeddings(model=EMBED_MODEL, prompt=chunk)["embedding"]
        collection.add(
            ids=[f"doc_{doc_id}"],
            embeddings=[embedding],
            documents=[chunk],
            metadatas=[{
                "ticker": record["ticker"],
                "kind": record["kind"],
                "title": record["title"],
                "date": record["date"],
                "source_url": record["source_url"],
                "chunk_index": i,
            }],
        )
        doc_id += 1

print(f"Done. Indexed {doc_id} chunks into Chroma collection '{COLLECTION_NAME}'")

Loaded 843 raw records (filings + news)


Embedding chunks: 100%|██████████| 843/843 [03:40<00:00,  3.82it/s]

Done. Indexed 908 chunks into Chroma collection 'nse_filings_news'


## 5. Query the RAG agent

Ask questions about your indexed filings + news. Prefix a query with a ticker
(e.g. `RELIANCE: ...`) to filter retrieval to just that stock.

**Note:** this agent explains and summarizes what's in the filings/news --
it does not give price predictions or investment advice, by design.

In [11]:
LLM_MODEL = "llama3.1:8b"
TOP_K = 6

SYSTEM_PROMPT = """You are a financial research assistant. Answer the user's
question about NSE-listed stocks using ONLY the provided context (filings and
news excerpts). If the context doesn't contain enough information, say so
clearly rather than guessing. Always mention which ticker(s) and dates your
answer draws from. Do not give investment advice or price predictions --
stick to summarizing and explaining what the context says."""


def retrieve(query: str, ticker_filter: str | None = None, k: int = TOP_K):
    query_embedding = ollama.embeddings(model=EMBED_MODEL, prompt=query)["embedding"]
    where = {"ticker": ticker_filter} if ticker_filter else None
    return collection.query(query_embeddings=[query_embedding], n_results=k, where=where)


def format_context(results) -> str:
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    blocks = []
    for doc, meta in zip(docs, metas):
        blocks.append(f"[{meta['ticker']} | {meta['kind']} | {meta['date']}] {meta['title']}\n{doc}")
    return "\n\n---\n\n".join(blocks)


def ask(question: str, ticker_filter: str | None = None) -> str:
    results = retrieve(question, ticker_filter)
    if not results["documents"][0]:
        return "No relevant filings or news found in the index for that query."

    context = format_context(results)
    user_prompt = f"Context:\n{context}\n\nQuestion: {question}"

    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response["message"]["content"]

In [12]:
# Try it out -- edit the question freely
answer = ask("What's driving RELIANCE stock this week?")
print(answer)

Based on the provided context, it appears that there are two main factors driving the RELIANCE stock this week:

1. The company's Annual General Meeting (AGM) which took place recently has had a positive impact on the stock price. On June 22nd, the stock jumped by 2% after the AGM (Moneycontrol.com).
2. There is also speculation about potential upside due to Reliance Industries' plans for Artificial Intelligence (AI), Jio IPO, and new business initiatives. Brokerages are predicting an upside of up to 34 percent on these developments (Moneycontrol.com).

Please note that the information provided is based solely on the given context and may not be a comprehensive analysis of the market factors affecting the stock price.


In [13]:
# Filtered to one ticker via the ticker_filter arg
answer = ask("Any recent board announcements?", ticker_filter="RELIANCE")
print(answer)

According to the filings, there have been several Board meetings and announcements made by Reliance recently:

* On January 16, 2026 (08-Jan-2026 20:17:21), a meeting was scheduled to consider and approve financial results for the quarter and nine months ended December 31, 2025.
* On October 17, 2025 (09-Oct-2025 19:13:17), a meeting was scheduled to consider and approve financial results for the quarter and half year ended September 30, 2025.
* On April 17, 2026 (24-Apr-2026 19:26:28), the Board approved Audited Financial Statements and Results for the FY ended March 31, 2026, and recommended a dividend of Rs. 6.00 per equity share.

Additionally, on July 10, 2026 (17:46:25), an upcoming meeting was announced to be held on July 17, 2026, to consider and approve financial results for the quarter ended June 30, 2026.


## Where this goes next

- **Hook into `nse-signal-engine`**: when a signal fires, call `ask()` with a prompt
  like *"Explain recent news/filings context for {ticker}"* and include that
  explanation in your ntfy.sh alert.
- **Swap Google News RSS for a paid news API** (NewsAPI, Polygon.io) for fuller
  article text instead of RSS summaries.
- **Schedule daily re-ingestion** (cron / `schedule` package) so the index stays current.


In [14]:
import sys
print(sys.executable)

c:\Python314\python.exe
